- how multiple attention heads operate in parallel and their outputs are combined to capture richer representations

- The core idea of the multi-head attention mechanism model is to calculate the representation capabilities of different positions in the input sequence through self-attention. It creates a set of self-attention layers, mainly including ‘scaled dot-product attention’ and ‘multi-head attention’, to handle inputs of varying sizes. Unlike CNNs, the multi-head attention mechanism uses stacked self-attention layers to process these inputs. The multi-head attention has significant advantages in handling the continuous state spaces. It is not limited by the temporal and spatial relationships between data and it is suitable for dealing with a series of agents.

- Unlike concatenated RNNs, it can achieve efficient layer outputs through parallel computations. It does not require multiple neural network layers to achieve output interaction between different stages of agents.

In [0]:
import torch
import torch.nn as nn
import math

class ScaledDotProductAttention(nn.Module):
    """
    Compute the scaled dot-product attention weights.
    """
    def __init__(self):
        super(ScaledDotProductAttention, self).__init__()

    def forward(self, Q, K, V, mask=None):
        # Q, K, V are of shape (batch_size, num_heads, seq_len, head_dim)
        
        # Calculate attention scores (Query * Key^T)
        # (batch_size, num_heads, seq_len, head_dim) @ (batch_size, num_heads, head_dim, seq_len)
        # -> (batch_size, num_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1))
        
        # Scale the scores by the square root of the key dimension (dk)
        dk = K.size()[-1]
        scores = scores / math.sqrt(dk)
        
        # Apply mask (if provided) to handle things like preventing attention to future tokens
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
            
        # Apply softmax to get attention weights
        attention_weights = torch.softmax(scores, dim=-1)
        
        # Multiply weights by Values to get the context vectors
        # (batch_size, num_heads, seq_len, seq_len) @ (batch_size, num_heads, seq_len, head_dim)
        # -> (batch_size, num_heads, seq_len, head_dim)
        output = torch.matmul(attention_weights, V)
        
        return output, attention_weights

class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention mechanism implementation.
    """
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads # Dimension of each head
        
        # Linear layers for Query, Key, and Value projections
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        self.attention = ScaledDotProductAttention()
        
        # Final output linear layer
        self.W_o = nn.Linear(d_model, d_model)
        
    def forward(self, Q, K, V, mask=None):
        # Q, K, V are of shape (batch_size, seq_len, d_model)
        batch_size = Q.size()[0]
        
        # 1. Apply linear projections and split into multiple heads
        # Result shape: (batch_size, seq_len, d_model) -> (batch_size, seq_len, num_heads, head_dim)
        # Transpose to (batch_size, num_heads, seq_len, head_dim) for batch matrix multiplication
        q = self.W_q(Q).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.W_k(K).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.W_v(V).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        
        # 2. Apply scaled dot-product attention
        context, attn_weights = self.attention(q, k, v, mask)
        
        # 3. Concatenate heads and apply final linear layer
        # Transpose back to (batch_size, seq_len, num_heads, head_dim)
        # Use contiguous() to ensure tensor is stored contiguously in memory before view
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        # Final linear projection
        output = self.W_o(context)
        
        return output, attn_weights

# --- Example Usage ---
d_model = 512 # The dimension of the input/output features
num_heads = 8 # Number of attention heads
seq_len = 10  # Sequence length
batch_size = 1

# Create dummy input data (e.g., token embeddings)
# Shape: (batch_size, seq_len, d_model)
input_data = torch.randn(batch_size, seq_len, d_model)

# Initialize the multi-head attention mechanism
multi_head_attn = MultiHeadAttention(d_model, num_heads)

# Forward pass
output_representation, attention_weights = multi_head_attn(input_data, input_data, input_data)

print(f"Input shape: {input_data.shape}")
print(f"Output shape: {output_representation.shape}") # Should be the same as input shape
print(f"Attention weights shape (per head): {attention_weights.shape}") # (batch_size, num_heads, seq_len, seq_len)
